# PHẦN 2: TẦNG TRUY XUẤT ỨNG VIÊN

In [ ]:
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'polars', 'faiss-gpu'])
import os
import pandas as pd
import numpy as np
import polars as pl
import torch
import torch.nn as nn
import scipy.sparse as sp
import faiss
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [ ]:
PROCESSED_DATA_DIR = '../data/processed'
TRAIN_PATH = os.path.join(PROCESSED_DATA_DIR, 'train_interactions.csv')
CAND_PATH  = os.path.join(PROCESSED_DATA_DIR, 'candidates_phase2.csv')
MAX_LEN   = 50
EMBED_DIM = 64

In [ ]:
# 1. SASRec
import polars as pl
df_train_pl = pl.scan_csv(TRAIN_PATH)
num_users = df_train_pl.select(pl.col('mapped_user_id').max()).collect()[0,0] + 1
num_items = df_train_pl.select(pl.col('mapped_item_id').max()).collect()[0,0] + 1

user_seqs = (df_train_pl
             .sort(["mapped_user_id", "timestamp"])
             .group_by("mapped_user_id", maintain_order=True)
             .agg(pl.col("mapped_item_id"))
             ).collect().to_pandas()

def pad_seq(seq):
    seq = list(seq)
    return seq[-MAX_LEN:] if len(seq) > MAX_LEN else [0]*(MAX_LEN - len(seq)) + seq

user_seqs['padded_seq'] = user_seqs['mapped_item_id'].apply(pad_seq)

class SASRec(nn.Module):
    def __init__(self, n_items, embed_dim, max_len):
        super().__init__()
        self.item_emb = nn.Embedding(n_items, embed_dim, padding_idx=0)
        self.pos_emb = nn.Embedding(max_len, embed_dim)
        layer = nn.TransformerEncoderLayer(d_model=embed_dim, nhead=1, batch_first=True)
        self.transformer = nn.TransformerEncoder(layer, num_layers=1)
    def forward(self, seqs):
        pos = torch.arange(seqs.size(1), device=seqs.device).unsqueeze(0).expand_as(seqs)
        mask = (seqs == 0)
        out = self.transformer(self.item_emb(seqs) + self.pos_emb(pos), src_key_padding_mask=mask)
        return out[:, -1, :]

model_sasrec = SASRec(num_items, EMBED_DIM, MAX_LEN).to(device).to(device)

# Real SASRec Training Loop
model_sasrec.train()
optimizer = torch.optim.Adam(model_sasrec.parameters(), lr=0.005)
criterion = nn.CrossEntropyLoss(ignore_index=0)

print("Đang khởi tạo Ma trận Sequence an toàn RAM...")
n_users_seq = len(user_seqs)
X_sas_train = np.zeros((n_users_seq, MAX_LEN), dtype=np.int32)
for idx, seq in enumerate(user_seqs['padded_seq']):
    X_sas_train[idx, :] = seq

epochs = 30
batch_size = 512
for ep in range(epochs):
    idx = np.random.permutation(len(X_sas_train))
    loss_ep = 0
    t_batches = 0
    for i in range(0, len(X_sas_train), batch_size):
        batch_seqs = torch.tensor(X_sas_train[idx[i:i+batch_size]])
        inputs = batch_seqs[:, :-1].to(device).to(device)
        targets = batch_seqs[:, -1].to(device).to(device)
        
        u_reps = model_sasrec(inputs)
        logits = torch.matmul(u_reps, model_sasrec.item_emb.weight.T)
        loss = criterion(logits, targets)
        
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        loss_ep += loss.item()
        t_batches += 1
    print(f"SASRec Epoch {ep+1}/{epochs} Loss: {loss_ep / t_batches:.4f}")
# --- FAISS SASRec ---
model_sasrec.eval()
print("Đang khởi tạo Ma trận Sequence an toàn RAM cho Inference...")
X_sas_np = np.zeros((len(user_seqs), MAX_LEN), dtype=np.int32)
for idx, seq in enumerate(user_seqs['padded_seq']):
    X_sas_np[idx, :] = seq
X_sas = torch.tensor(X_sas_np)
with torch.no_grad():
    i_embs_tensor = model_sasrec.item_emb.weight[1:]
    i_embs = torch.nn.functional.normalize(i_embs_tensor, p=2, dim=1).cpu().numpy()
    
    index_sas = faiss.IndexFlatIP(EMBED_DIM)
    index_sas.add(i_embs)
    all_top_idx = []
    chunk_size = 5000
    for i in range(0, len(X_sas), chunk_size):
        u_reps_tensor = model_sasrec(X_sas[i:i+chunk_size].to(device))
        u_reps = torch.nn.functional.normalize(u_reps_tensor, p=2, dim=1).cpu().numpy()
        _, idx = index_sas.search(u_reps, min(200, num_items - 1))
        all_top_idx.append(idx)
    top_idx = np.vstack(all_top_idx)

n_users_sas = len(user_seqs['mapped_user_id'])
n_cands = top_idx.shape[1]

u_arr = np.repeat(user_seqs['mapped_user_id'].values, n_cands)
i_arr = top_idx.flatten() + 1
r_arr = np.tile(np.arange(1, n_cands + 1), n_users_sas)

df_sasrec = pd.DataFrame({
    'mapped_user_id': u_arr, 
    'mapped_item_id': i_arr, 
    'sasrec_rank': r_arr.astype('float32')
})
print("SASRec Inference Done.")


In [ ]:
# 2. LightGCN / FAISS
df_train_sub = df_train_pl.select(['mapped_user_id', 'mapped_item_id']).collect()
u_idx, i_idx = df_train_sub['mapped_user_id'].to_numpy(), df_train_sub['mapped_item_id'].to_numpy()
adj = sp.coo_matrix((np.ones(len(u_idx)), (u_idx, i_idx + num_users)), shape=(num_users+num_items, num_users+num_items))
adj = adj + adj.T.multiply(adj.T > adj) - adj.multiply(adj.T > adj)

d_inv = np.power(np.array(adj.sum(1)), -0.5).flatten()
d_inv[np.isinf(d_inv)] = 0.
d_mat = sp.diags(d_inv)
norm_adj = d_mat.dot(adj).dot(d_mat).tocoo()

indices = torch.LongTensor(np.vstack((norm_adj.row, norm_adj.col)))
norm_adj_t = torch.sparse_coo_tensor(indices, torch.FloatTensor(norm_adj.data), norm_adj.shape).to(device).to(device)

class LightGCN(nn.Module):
    def __init__(self, u, i, dim):
        super().__init__()
        self.u_emb = nn.Embedding(u, dim)
        self.i_emb = nn.Embedding(i, dim)
    def forward(self, adj):
        emb = torch.cat([self.u_emb.weight, self.i_emb.weight])
        all_embs = [emb]
        for _ in range(2):
            emb = torch.sparse.mm(adj, emb)
            all_embs.append(emb)
        final = torch.stack(all_embs, dim=1).mean(dim=1)
        return torch.split(final, [num_users, num_items])

model_lgcn = LightGCN(num_users, num_items, EMBED_DIM).to(device).to(device)

# Real LightGCN Training Loop
model_lgcn.train()
optimizer = torch.optim.Adam(model_lgcn.parameters(), lr=0.001)

pos_pairs = df_train_pl.select(['mapped_user_id', 'mapped_item_id']).collect().to_numpy()
epochs = 30
batch_size = 20480

for ep in range(epochs):
    np.random.shuffle(pos_pairs)
    loss_ep = 0
    t_batches = 0
    
    for i in range(0, len(pos_pairs), batch_size):
        batch = pos_pairs[i:i+batch_size]
        users = batch[:, 0]
        pos_items = batch[:, 1]
        neg_items = np.random.randint(1, num_items, size=len(users))
        
        u_embs_all, i_embs_all = model_lgcn(norm_adj_t)
        
        u_emb = u_embs_all[users]
        p_emb = i_embs_all[pos_items]
        n_emb = i_embs_all[neg_items]
        
        pos_scores = (u_emb * p_emb).sum(1)
        neg_scores = (u_emb * n_emb).sum(1)
        
        loss_bpr = -torch.log(torch.sigmoid(pos_scores - neg_scores) + 1e-8).mean()
        u_emb_0 = model_lgcn.u_emb.weight[users]
        p_emb_0 = model_lgcn.i_emb.weight[pos_items]
        n_emb_0 = model_lgcn.i_emb.weight[neg_items]
        reg_loss = (1/2)*(u_emb_0.norm(2).pow(2) + p_emb_0.norm(2).pow(2) + n_emb_0.norm(2).pow(2)) / float(len(users))
        loss = loss_bpr + 1e-4 * reg_loss
        
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        loss_ep += loss_bpr.item()
        t_batches += 1
    print(f"LightGCN Epoch {ep+1}/{epochs} BPR Loss: {loss_ep / t_batches:.4f}")
# --- FAISS LightGCN ---
model_lgcn.eval()
with torch.no_grad():
    u_embs_raw, i_embs_raw = model_lgcn(norm_adj_t)
    u_embs = torch.nn.functional.normalize(u_embs_raw, p=2, dim=1).cpu().numpy()
    i_embs = torch.nn.functional.normalize(i_embs_raw, p=2, dim=1).cpu().numpy()

index = faiss.IndexFlatIP(EMBED_DIM)
index.add(i_embs[1:])

all_lgcn_idx = []
chunk_size = 5000
print("Bắt đầu quét FAISS cho LightGCN...")
for i in range(0, len(u_embs), chunk_size):
    _, idx = index.search(u_embs[i:i+chunk_size], min(200, num_items - 1))
    all_lgcn_idx.append(idx)
indices_faiss = np.vstack(all_lgcn_idx)

n_users_lgcn = len(indices_faiss)
n_cands = indices_faiss.shape[1]
u_arr_lgcn = np.repeat(np.arange(n_users_lgcn), n_cands)
i_arr_lgcn = indices_faiss.flatten() + 1
r_arr_lgcn = np.tile(np.arange(1, n_cands + 1), n_users_lgcn)

df_lightgcn = pd.DataFrame({
    'mapped_user_id': u_arr_lgcn, 
    'mapped_item_id': i_arr_lgcn, 
    'lightgcn_rank': r_arr_lgcn.astype('float32')
})
print("LightGCN Inference Done.")


In [ ]:
# 3. Union Scale 

print("Đang chuyển đổi sang Polars để ghép bảng siêu tốc...")
pl_sasrec = pl.from_pandas(df_sasrec)
pl_lightgcn = pl.from_pandas(df_lightgcn)

# Kích hoạt bộ dọn rác, XÓA NGAY Pandas Dataframe để làm trống RAM Colab
del df_sasrec, df_lightgcn
gc.collect()

print("Đang Outer Join bằng Backend Rust...")
# Dùng coalesce=True để gộp 2 cột key lại làm 1 (Tương đương how='outer' của Pandas)
df_union = pl_sasrec.join(pl_lightgcn, on=['mapped_user_id', 'mapped_item_id'], how='full', coalesce=True)

print("Đang lưu Candidates...")
df_union.write_csv(CAND_PATH)

# Dọn dẹp
del pl_sasrec, pl_lightgcn, df_union
gc.collect()
print("Candidates Phase 2 saved without 999 FillNA (RAM Safe).")

Candidates Phase 2 saved without 999 FillNA.


In [6]:
# 4. Candidate Recall@100 Pipeline Check
df_test = pd.read_csv(os.path.join(PROCESSED_DATA_DIR, 'test_interactions.csv'))
truth = df_test.groupby('mapped_user_id')['mapped_item_id'].apply(set).to_dict()

# Kiểm tra xem Ground Truth có nằm trong Candidates không
cands_grouped = df_union.groupby('mapped_user_id')['mapped_item_id'].apply(set).to_dict()
hit_in_recall = 0
valid_u = 0

for u, true_items in truth.items():
    if u in cands_grouped:
        valid_u += 1
        if len(true_items.intersection(cands_grouped[u])) > 0:
            hit_in_recall += 1

print(f"Candidate Recall@100: {hit_in_recall / valid_u if valid_u > 0 else 0:.4f}")

Candidate Recall@100: 0.1014
